# Final detector evaluation - dual track

Six detectors judge each (source, summary) pair. The detector sees **only a source and the summary**, never which one it got or that anything was edited.

| Track | Pair | Gold |
|---|---|---|
| A | original source + summary | 0, faithful |
| B | corrupted source + summary | 1, hallucinated |

**Reasoning is ON.** The `reasoning` field is omitted so each model uses its default. `max_tokens` is 2048 because reasoning tokens count against the output budget - at 64 a reasoning model spends the whole budget thinking and returns nothing.

**Output is a bare `0` or `1`.** Measured: this saves about 2% of cost, not more, because with reasoning on the thinking tokens dominate and they are generated before the answer. The real reason for it is reliability - Qwen and Llama returned prose averaging 96 and 22 tokens in the pilot, and every unparseable verdict came from those two.

**Raw responses are not stored.** Only rows whose answer could not be parsed are written to a separate small file, so failures stay diagnosable without carrying prose for every row.

**Scope: the whole corpus**, all three splits. 11,718 items across 6 detectors is 70,308 calls, roughly $32 and several hours. Checkpointing every 50 calls means a kill costs at most 50.

In [ ]:
import os
import re
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(os.path.abspath("../../.env"))
api_keys = [k for k in (os.getenv("OPENROUTER_API_KEY_NEW"), os.getenv("OPENROUTER_API_KEY")) if k]
if not api_keys:
    raise ValueError("No API key found! Check the .env file at the repository root.")

DETECTORS = {
    "gpt_4o_mini":           "openai/gpt-4o-mini",
    "deepseek_v4_flash":     "deepseek/deepseek-v4-flash",
    "gemini_3_1_flash_lite": "google/gemini-3.1-flash-lite",
    "llama_3_1_70b":         "meta-llama/llama-3.1-70b-instruct",
    "qwen_2_5_72b":          "qwen/qwen-2.5-72b-instruct",
    "grok_4_3":              "x-ai/grok-4.3",
}

SPLITS = ("train", "valid", "test")   # whole corpus; ("test",) for just the test split
OUT_DIR = "../../evaluation_summarization"
TAG = "_".join(SPLITS)
BACKUP_FILE = f"backup_final_verdicts_{TAG}.csv"
UNPARSED_FILE = f"unparsed_verdicts_{TAG}.csv"
CHECKPOINT_EVERY = 50
MAX_WORKERS = 12
MAX_TOKENS = 2048

os.makedirs(OUT_DIR, exist_ok=True)
print(f"{len(api_keys)} key(s). {len(DETECTORS)} detectors. Splits: {SPLITS}")

In [ ]:
corrupted = pd.read_csv("final_luna_corrupted.csv")
track_a = pd.read_csv("final_luna_track_a.csv")
corrupted = corrupted[corrupted["split"].isin(SPLITS)]
track_a = track_a[track_a["split"].isin(SPLITS)]

items = []
for _, r in track_a.iterrows():
    items.append({"item_id": f"A_{r['doc_id']}", "track": "A", "gold": 0,
                  "doc_id": r["doc_id"], "split": r["split"], "pipeline": "",
                  "method": "", "n": 0, "target_categories": "",
                  "source": r["source_original"], "summary": r["summary"]})
for _, r in corrupted.iterrows():
    items.append({"item_id": f"B_{r['item_id']}", "track": "B", "gold": 1,
                  "doc_id": r["doc_id"], "split": r["split"], "pipeline": r["pipeline"],
                  "method": r["method"], "n": r["n"],
                  "target_categories": r["target_categories"],
                  "source": r["source_corrupted"], "summary": r["summary"]})

items = pd.DataFrame(items)
assert items["item_id"].is_unique, "duplicate item_id - jobs would be double-paid"
assert items["source"].notna().all() and items["summary"].notna().all(), "blank source or summary"

n_calls = len(items) * len(DETECTORS)
print(f"{len(items)} items: {(items['track'] == 'A').sum()} Track A, "
      f"{(items['track'] == 'B').sum()} Track B")
print(f"{n_calls:,} detector calls")
print("")
# measured in the pilot, reasoning ON, per item through all six detectors
print(f"estimated cost: ${len(items) * 0.00274:.2f}")
print(f"   of which grok_4_3: ${len(items) * 0.00196:.2f} "
      f"({100 * 0.00196 / 0.00274:.0f}% of the bill)")

In [ ]:
DETECT_PROMPT = """You are a medical verification assistant.

Determine whether the summary contains information that is not supported by, or that contradicts, the source document.

Judge only against the source document. Do not use outside medical knowledge. A summary that is incomplete but contains nothing unsupported is NOT hallucinated.

Source document:
{source}

Summary:
{summary}

Answer with a single character and nothing else:
1 if the summary contains hallucinated information
0 if it does not

Do not explain. Do not write anything except the digit.
"""

key_lock = threading.Lock()
active_key_index = 0


def call_model(prompt, model_name, max_retries=3):
    """No `reasoning` field: each model uses its default, so the reasoners reason.
    MAX_TOKENS must exceed the reasoning budget or they return an empty answer."""
    global active_key_index
    payload = {"model": model_name, "messages": [{"role": "user", "content": prompt}],
               "temperature": 0.0, "max_tokens": MAX_TOKENS}
    for attempt in range(max_retries):
        for _ in range(len(api_keys)):
            with key_lock:
                k = api_keys[active_key_index]
            try:
                r = requests.post(
                    "https://openrouter.ai/api/v1/chat/completions",
                    headers={"Authorization": f"Bearer {k}", "Content-Type": "application/json"},
                    json=payload, timeout=180)
            except requests.RequestException:
                break
            if r.status_code == 200:
                body = r.json()
                usage = body.get("usage") or {}
                detail = usage.get("completion_tokens_details") or {}
                return (body["choices"][0]["message"]["content"],
                        detail.get("reasoning_tokens", 0), usage.get("cost", 0.0))
            if r.status_code in (401, 402, 403, 429):
                with key_lock:
                    active_key_index = (active_key_index + 1) % len(api_keys)
                continue
            break
        time.sleep(2 * (attempt + 1))
    return None, 0, 0.0


def parse_verdict(text):
    """1, 0, or None. Accepts a bare digit first, then the older labelled form."""
    if not text:
        return None
    s = text.strip()
    if s in ("0", "1"):
        return int(s)
    m = re.search(r"ANSWER:\s*([01])", s)
    if m:
        return int(m.group(1))
    m = re.search(r"\b([01])\b", s)
    return int(m.group(1)) if m else None


def run_one(task):
    item, name, slug = task
    raw, reasoning_tokens, cost = call_model(
        DETECT_PROMPT.format(source=item["source"], summary=item["summary"]), slug)
    verdict = parse_verdict(raw)
    return {"job_key": f"{name}|{item['item_id']}", "detector": name,
            "item_id": item["item_id"], "doc_id": item["doc_id"], "split": item["split"],
            "track": item["track"], "gold": item["gold"], "pipeline": item["pipeline"],
            "method": item["method"], "n": item["n"],
            "target_categories": item["target_categories"],
            "prediction": verdict,
            "correct": None if verdict is None else int(verdict == item["gold"]),
            "reasoning_tokens": reasoning_tokens, "cost": cost,
            # kept only when unparseable, dropped from the saved table otherwise
            "raw_response": None if verdict is not None else raw}


tasks = [(row, name, slug) for _, row in items.iterrows() for name, slug in DETECTORS.items()]
assert len({f"{n}|{r['item_id']}" for r, n, _ in tasks}) == len(tasks), "duplicate job_key"
print(f"{len(tasks):,} calls queued.")

In [ ]:
# PRE-FLIGHT. One call per detector before committing to the rest.
# Watch reason_tok against MAX_TOKENS: if a model approaches it, the answer gets
# truncated and the call is wasted.
probe = items.iloc[0]
smoke_ok = True
print(f"{'':>6}  {'detector':<24} {'verdict':>7} {'reason_tok':>11}  cost")
for name, slug in DETECTORS.items():
    raw, rt, cost = call_model(
        DETECT_PROMPT.format(source=probe["source"], summary=probe["summary"]), slug)
    verdict = parse_verdict(raw)
    if verdict is None:
        smoke_ok = False
    print(f"{'OK' if verdict is not None else 'FAILED':>6}  {name:<24} "
          f"{str(verdict):>7} {rt:>11}  ${cost:.5f}")

if not smoke_ok:
    raise RuntimeError("Pre-flight failed. Do not run the full evaluation.")
print("")
print("Pre-flight passed.")

In [ ]:
done = {}
if os.path.exists(BACKUP_FILE):
    prior = pd.read_csv(BACKUP_FILE)
    done = {r["job_key"]: r for _, r in prior.iterrows()}
    print(f"Resuming: {len(done):,} verdicts already collected.")
else:
    print("No backup - starting fresh.")

pending = [t for t in tasks if f"{t[1]}|{t[0]['item_id']}" not in done]
print(f"{len(pending):,} of {len(tasks):,} calls still to make.")
print("")

start = time.time()
results = [done[f"{n}|{r['item_id']}"].to_dict()
           for r, n, _ in tasks if f"{n}|{r['item_id']}" in done]
lock = threading.Lock()


def checkpoint():
    pd.DataFrame(results).to_csv(BACKUP_FILE, index=False, encoding="utf-8-sig")


if pending:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        for finished, record in enumerate(pool.map(run_one, pending), start=1):
            with lock:
                results.append(record)
                if finished % CHECKPOINT_EVERY == 0 or finished == len(pending):
                    checkpoint()
                    usable = sum(r.get("prediction") is not None for r in results)
                    spent = sum(float(r.get("cost") or 0) for r in results)
                    print(f"   --- saved at {finished:,}/{len(pending):,} "
                          f"(usable={usable:,}, ${spent:.3f}, {time.time() - start:.0f}s) ---")

checkpoint()
v = pd.DataFrame(results)
print("")
print(f"{len(v):,} verdicts, {int(v['prediction'].notna().sum()):,} usable, "
      f"${v['cost'].astype(float).sum():.3f}")

In [ ]:
# Unparseable rows keep their raw text, in their own small file.
bad = v[v["prediction"].isna()]
if len(bad):
    bad[["detector", "item_id", "track", "raw_response"]].to_csv(
        os.path.join(OUT_DIR, UNPARSED_FILE), index=False, encoding="utf-8-sig")
    print(f"{len(bad)} unparseable -> {UNPARSED_FILE}")
    print(bad.groupby("detector").size().to_string())
else:
    print("no unparseable verdicts")

# One CSV per detector, no raw column.
COLS = ["item_id", "doc_id", "split", "track", "gold", "pipeline", "method", "n",
        "target_categories", "prediction", "correct", "reasoning_tokens"]
print("")
print(f"{'file':<66} {'rows':>6} {'acc%':>6}")
print("-" * 80)
for name in DETECTORS:
    d = v[v["detector"] == name][COLS]
    path = os.path.join(OUT_DIR, f"final_{TAG}_evaluated_by_{name}.csv")
    d.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"{os.path.basename(path):<66} {len(d):>6} "
          f"{100 * d['correct'].dropna().mean():>6.1f}")

In [ ]:
scored = v.dropna(subset=["prediction"]).copy()
scored["correct"] = scored["correct"].astype(int)

rows = []
for name in DETECTORS:
    d = scored[scored["detector"] == name]
    a = d[d["track"] == "A"]["correct"].mean() * 100
    b = d[d["track"] == "B"]["correct"].mean() * 100
    rows.append({"Model": name, "Correct": round(a, 2), "Wrong": round(b, 2),
                 "Overall": round(100 * d["correct"].mean(), 2),
                 "balanced_err": round(((100 - a) + (100 - b)) / 2, 2),
                 "cost": round(d["cost"].astype(float).sum(), 3)})
summary = pd.DataFrame(rows).sort_values("balanced_err")
summary.to_csv(os.path.join(OUT_DIR, f"final_{TAG}_accuracy.csv"),
               index=False, encoding="utf-8-sig")
print("Correct = original source. Wrong = corrupted source. Both are accuracies.")
print("")
print(summary.to_string(index=False))
print("")
print("Run build_accuracy_table.py for the category and method breakdowns.")